# 📓 Semana 11 · Dia 3 — Embeddings e chunking de documentos

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | GenAI Engineer Associate |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Pipeline de chunking + embeddings rodando |

---


## 📖 Teoria — Embeddings

Um **embedding** é a representação numérica (vetor de N dimensões) do significado de um texto. Textos similares ficam **próximos** no espaço vetorial — a busca semântica usa essa distância.

- Modelos de embedding: BGE, GTE, OpenAI text-embedding (via FMA)
- Similaridade: cosseno (1 = idêntico; 0 = ortogonal)
- **Chunking**: dividir documentos grandes em blocos (chunks) de ~200–800 tokens com overlap — o modelo só vê o contexto da janela.


## 📖 Teoria — Estratégias de chunking

| Estratégia | Como | Quando |
|---|---|---|
| **Fixo** | N tokens com overlap | docs simples |
| **Por estrutura** | por parágrafo/seção | docs com markdown/títulos |
| **Semântico** | junta frases similares | docs longos e heterogêneos |

Regra: chunk pequeno demais = sem contexto; grande demais = ruído. Teste tamanho/overlap com as métricas de avaliação (Semana 12).


### 💻 Na prática — Chunking na prática

Crie o pipeline de chunking dos produtos do catálogo.


In [ ]:
# Documentos do projeto: descrições de produtos
docs = spark.sql("SELECT StockCode, Description FROM workspace.prata.dim_produto WHERE Description IS NOT NULL")\
    .limit(500).toPandas()
print("Documentos:", len(docs))

In [ ]:
# Chunking por estrutura simples (cada produto = 1 doc)
from pyspark.sql.functions import concat, lit, col
docs_spark = spark.table("workspace.prata.dim_produto")\
    .filter(col("Description").isNotNull())\
    .limit(500)\
    .withColumn("doc", concat(lit("Produto: "), col("Description"),
                                lit(" | Código: "), col("StockCode")))
print("Docs prontos para embedding:", docs_spark.count())

In [ ]:
# Texto de exemplo para embedding
textos = ["copo de vidro vermelho", "taça para vinho", "teclado mecânico"]
print(textos)

### 💻 Na prática — Gerando embeddings

Use a FMA de embeddings (ajuste ao endpoint disponível) e meça similaridade.


In [ ]:
# Embeddings via FMA
from mlflow.deployments import get_deploy_client
client = get_deploy_client("databricks")
resp = client.predict(
    endpoint="databricks-bge-large-en",
    inputs={"input": textos})
vetores = resp["data"]
print("Dimensão do embedding:", len(vetores[0]["embedding"]))
print("Nº de vetores:", len(vetores))

In [ ]:
# Similaridade por cosseno (sem numpy, puro Python)
import math
def cosseno(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    na = math.sqrt(sum(x*x for x in a)); nb = math.sqrt(sum(y*y for y in b))
    return dot / (na * nb)
v = [d["embedding"] for d in vetores]
print("copo x taça (semelhantes):", round(cosseno(v[0], v[1]), 3))
print("copo x teclado (diferentes):", round(cosseno(v[0], v[2]), 3))

> 🎯 **Dica de prova**: GenAI Assoc (Data Prep ~20%): chunking e embeddings são o coração. Pergunta típica: 'por que dividir documentos em chunks?' → janela de contexto + relevância do trecho.


## 🎯 Exercícios de fixação

**1.** Por que o tamanho do chunk importa para a qualidade da resposta?

**2.** O que significa similaridade de cosseno = 1? E = 0?

**3.** Teste 2 tamanhos de chunk e anote qual parece melhor.


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Chunk

Pequeno = contexto insuficiente; grande = ruído e tokens demais. O sweet spot (~200-800 tokens) depende do documento.

**2.** Cosseno

1 = mesma direção (muito similares); 0 = ortogonais (sem relação); negativo = opostos.

**3.** Teste

Compare respostas para a mesma pergunta com chunks de 100 vs 500 tokens — a avaliação da Semana 12 quantifica.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*